# AG12 · Coding Agents

**Track:** Agents (lesson 12 of 15) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free (the scripted trace really edits files and really runs `pytest`), or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** AG02 (agent loop), AG03 (tool design), AG09 (permissions & sandboxing) · **You will build:** a mini coding agent (read / grep / edit / run-tests) and a SWE-bench-style resolve-rate harness

---
## 🎯 Why this matters
Coding is the first domain where agents do real, measurable work, for one reason: **code comes with a verifier**. Tests tell the agent (and you) whether the
last edit helped. That feedback loop — act, run tests, read failures, act again — is what separates a coding agent from autocomplete, and it is a template for
every agentic domain where you can build a checker.

**Real scenario.** *Kōwhai Retail* maintains a small internal library, `shopkit` (pricing, cart totals, pagination). Three bug reports arrive from the storefront team.
Can an agent fix them without touching the tests, without wandering outside the repo, and — crucially — can *you prove* it did, independently of what it says?

## 🧭 Learning objectives
By the end you can:
1. Design a minimal **tool surface** for code (read, search, exact-match edit, run tests) and explain why each tool is shaped the way it is.
2. Build a **sandbox** that blocks path escapes (`..`, absolute paths, symlinks, sibling-prefix tricks) and protects the tests from edits.
3. Run a **test-driven agent loop** whose success is verified by `pytest` in a subprocess, not by the agent's own claim.
4. Compute a SWE-bench-style **resolve rate** (fail-to-pass *and* pass-to-pass) and see why an agent's "done!" is not evidence.
5. Map what you built onto the **Claude Agent SDK, skills, hooks and subagents** at a conceptual level.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: why code is special, and what the agent needs

```
  issue text ─▶ [ model ] ─▶ grep / read_file ─▶ edit_file ─▶ run_tests ─┐
                   ▲                                                     │ failures / PASS
                   └─────────────────────────────────────────────────────┘
  verified OUTSIDE the agent:  fresh copy of the tests  ─▶  pytest  ─▶  resolved? (yes/no)
```

| Design choice | Why |
|---|---|
| **Few, sharp tools**: read, grep, edit, run tests | Each maps to something an engineer does; fewer choices means fewer wrong ones (AG03). |
| **Edit by exact string replacement**, not "rewrite the file" | A diff-sized change is reviewable, cheap in tokens, and cannot silently drop code the model forgot to re-emit. The edit *fails loudly* if the target text is missing or ambiguous. |
| **Numbered line reads, ranges** | The model can ask for 40 lines instead of a 2 000-line file: context is the scarce resource (AG05). |
| **Repo context up front** (a file tree in the prompt) | Saves exploratory turns; grep does the rest. |
| **Sandboxing** | The agent must only touch the project directory. File tools check paths; running arbitrary code needs an OS-level boundary (container/VM) — see §4. |
| **Tests are read-only** | Otherwise the cheapest way to "pass" is to edit the test. Protect them *and* re-copy pristine tests before grading. |
| **Independent grading** | The agent's final message is a claim; the test run is evidence. |

**Evaluation, SWE-bench style.** Each task is `(repo at a buggy commit, issue text, tests)`. Tests split into **FAIL_TO_PASS** (fail before the fix; must pass after) and
**PASS_TO_PASS** (passed before; must *still* pass). A task is **resolved** only if both groups pass. The **resolve rate** is resolved tasks / total tasks.
(SWE-bench uses real GitHub issues from open-source repositories; we use a tiny synthetic version so it runs in seconds. See Further reading.)

## 2 · The world: a small library with three seeded bugs
We keep the *correct* source of `shopkit` and its tests as strings. A **task** is defined by one *mutation* (correct text → buggy text) and an issue written the way a
colleague would write it — notice it does **not** say which file is wrong. Everything is created inside a fresh temporary directory; nothing touches this repo.

In [ ]:
import os, re, sys, json, shutil, tempfile, pathlib, subprocess, difflib, textwrap
from dataclasses import dataclass
from agentkit import tool, ToolBox, get_llm, say, call, calls, run_agent, ReplayLLM

FILES = {
"shopkit/__init__.py": "",
"shopkit/pricing.py": '''\
def apply_discount(price: float, pct: float) -> float:
    """Return price after a percentage discount (pct is 0-100)."""
    if not 0 <= pct <= 100:
        raise ValueError("pct must be between 0 and 100")
    return round(price * (1 - pct / 100), 2)


def add_gst(price: float, rate: float = 0.15) -> float:
    """Add NZ GST (15%) to a GST-exclusive price."""
    return round(price * (1 + rate), 2)
''',
"shopkit/cart.py": '''\
from dataclasses import dataclass


@dataclass
class Item:
    sku: str
    price: float
    qty: int = 1


def cart_total(items) -> float:
    """Total price of the cart: the sum of price * qty over every line."""
    return round(sum(i.price * i.qty for i in items), 2)


def count_units(items) -> int:
    """Total number of units across all lines."""
    return sum(i.qty for i in items)
''',
"shopkit/paging.py": '''\
def paginate(items, page: int, size: int) -> list:
    """Return the items on a 1-indexed page."""
    if page < 1 or size < 1:
        raise ValueError("page and size must be >= 1")
    start = (page - 1) * size
    return list(items[start:start + size])


def page_count(n_items: int, size: int) -> int:
    """Number of pages needed for n_items."""
    return -(-n_items // size)
''',
}

TESTS = {
"tests/test_pricing.py": '''\
import pytest
from shopkit.pricing import apply_discount, add_gst


def test_discount_quarter():
    assert apply_discount(200.0, 25) == 150.0


def test_discount_zero_is_identity():
    assert apply_discount(100.0, 0) == 100.0


def test_discount_rejects_over_100():
    with pytest.raises(ValueError):
        apply_discount(100.0, 120)


def test_gst():
    assert add_gst(100.0) == 115.0
''',
"tests/test_cart.py": '''\
from shopkit.cart import Item, cart_total, count_units


def test_total_respects_quantity():
    assert cart_total([Item("TEE", 10.0, 3), Item("CAP", 5.5, 2)]) == 41.0


def test_total_single_unit():
    assert cart_total([Item("TEE", 10.0)]) == 10.0


def test_count_units():
    assert count_units([Item("TEE", 10.0, 3), Item("CAP", 5.5, 2)]) == 5
''',
"tests/test_paging.py": '''\
import pytest
from shopkit.paging import paginate, page_count

ITEMS = [1, 2, 3, 4, 5]


def test_first_page():
    assert paginate(ITEMS, 1, 2) == [1, 2]


def test_last_partial_page():
    assert paginate(ITEMS, 3, 2) == [5]


def test_page_count():
    assert page_count(5, 2) == 3


def test_rejects_page_zero():
    with pytest.raises(ValueError):
        paginate(ITEMS, 0, 2)
''',
}

@dataclass
class Task:
    id: str
    issue: str
    file: str          # where the bug lives (hidden from the agent)
    correct: str       # the correct text
    buggy: str         # what the mutation puts there instead
    symbol: str        # what a good engineer would grep for

TASKS = [
    Task("SK-101", "Storefront: a 25% discount on a $200 item gives a negative price. Expected $150.00. Seems to affect every non-zero discount.",
         "shopkit/pricing.py", "price * (1 - pct / 100)", "price * (1 - pct / 10)", "def apply_discount"),
    Task("SK-102", "Cart total is wrong when a customer buys more than one of something: 3 tees at $10 and 2 caps at $5.50 should be $41.00 but we show a lower number.",
         "shopkit/cart.py", "sum(i.price * i.qty for i in items)", "sum(i.price for i in items)", "def cart_total"),
    Task("SK-103", "Pagination skips the first page of results and the last partial page comes back empty. Page 1 of 5 items with size 2 should be [1, 2].",
         "shopkit/paging.py", "start = (page - 1) * size", "start = page * size", "def paginate"),
]

def build_repo(task: Task, parent: pathlib.Path) -> pathlib.Path:
    root = parent / task.id
    for rel, text in {**FILES, **TESTS}.items():
        p = root / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text)
    src = (root / task.file).read_text()
    assert src.count(task.correct) == 1, "mutation target must be unique"
    (root / task.file).write_text(src.replace(task.correct, task.buggy))
    return root

WORK = tempfile.TemporaryDirectory(prefix="ag12_")
WORKDIR = pathlib.Path(WORK.name).resolve()
print("scratch area:", WORKDIR)
assert pathlib.Path.cwd().resolve() not in WORKDIR.parents, "scratch repos must live outside the course repo"

## 3 · The verifier: run `pytest` in a subprocess
This function is the ground truth of the whole lesson. It runs `python -m pytest` with the repo as working directory, a **timeout**, a **minimal environment** (we do not hand the
child process our whole environment — it could contain secrets), and parses per-test outcomes from pytest's `-rA` summary. The agent's `run_tests` tool and the final grader both use it.

In [ ]:
PYTEST_LINE = re.compile(r"^(PASSED|FAILED|ERROR)\s+(\S+)", re.M)

def pytest_run(root: pathlib.Path, target: str = "", timeout: int = 60):
    """Returns (returncode, output, {test_id: 'PASSED'|'FAILED'|'ERROR'})."""
    env = {k: os.environ[k] for k in ("PATH", "HOME", "SYSTEMROOT") if k in os.environ}
    env.update(PYTHONDONTWRITEBYTECODE="1", PYTHONPATH=str(root), PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")   # no third-party plugins: faster and reproducible
    cmd = [sys.executable, "-m", "pytest", "-q", "--tb=short", "-rA", "-p", "no:cacheprovider"] + ([target] if target else [])
    try:
        p = subprocess.run(cmd, cwd=root, env=env, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return 124, f"pytest timed out after {timeout}s", {}
    out = p.stdout + p.stderr
    return p.returncode, out, {m.group(2): m.group(1) for m in PYTEST_LINE.finditer(out)}

# Baseline: every task's repo must start RED, and only because of its own bug
BASE = {}
for t in TASKS:
    root = build_repo(t, WORKDIR)
    rc, out, res = pytest_run(root)
    f2p = sorted(k for k, v in res.items() if v != "PASSED")
    p2p = sorted(k for k, v in res.items() if v == "PASSED")
    BASE[t.id] = dict(f2p=f2p, p2p=p2p)
    print(f"{t.id}: rc={rc}  FAIL_TO_PASS={len(f2p)}  PASS_TO_PASS={len(p2p)}   e.g. {f2p[0].split('::')[-1]}")
    assert rc != 0 and f2p, "a seeded bug must make at least one test fail"
    assert all(len(v) > 0 for v in BASE[t.id].values()), "every task needs both groups to be meaningful"

Note the two groups per task. `test_discount_zero_is_identity` passes *even with the bug* (a 0% discount hides it) — it belongs to PASS_TO_PASS, and is exactly the kind of test a
careless "fix" can break. A task is resolved only when **both** groups are green.

## 4 · Build it: a sandbox and four tools
**Path-escape protection.** Every path the model gives us is *untrusted input*. We resolve it against the repo root, following symlinks, and then check the result is still inside the root
using `Path.relative_to` — **not** `str.startswith`, which wrongly accepts a sibling directory like `repo_evil` for root `repo`. Absolute paths and `..` fall out of the same check.

**Protected paths.** Edits under `tests/` are refused. **Exact edits.** `edit_file(path, old, new)` requires `old` to occur *exactly once* — zero or many matches is an error the model can read and recover from.

**Honest limit.** These checks govern our *file tools*. `run_tests` executes the project's code as your OS user; in production that belongs in a container or VM with no network and no secrets (AG09).

In [ ]:
class Sandbox:
    PROTECTED = ("tests",)

    def __init__(self, root):
        self.root = pathlib.Path(root).resolve()

    def resolve(self, rel: str) -> pathlib.Path:
        p = (self.root / rel).resolve()            # an absolute `rel` replaces root here; symlinks are followed
        try:
            p.relative_to(self.root)
        except ValueError:
            raise PermissionError(f"path {rel!r} is outside the project directory; use paths relative to the repo root") from None
        return p

    def writable(self, rel: str) -> pathlib.Path:
        p = self.resolve(rel)
        parts = p.relative_to(self.root).parts
        if parts and parts[0] in self.PROTECTED:
            raise PermissionError("tests/ is read-only: fix the code under test, not the tests")
        if p.suffix != ".py":
            raise PermissionError("only .py files can be edited")
        return p

SB = {"sb": None}          # the active sandbox (set per task)

def _sb() -> Sandbox:
    return SB["sb"]

@tool
def read_file(path: str, start: int = 1, end: int = 0) -> str:
    """Read a file with line numbers. Use start/end to read a range instead of the whole file.

    Args:
        path: File path relative to the repo root, e.g. "shopkit/cart.py".
        start: First line to show (1-indexed).
        end: Last line to show; 0 means up to 200 lines from start.
    """
    p = _sb().resolve(path)
    lines = p.read_text().splitlines()
    end = end or min(len(lines), start + 199)
    return "\n".join(f"{i:>4}  {lines[i-1]}" for i in range(max(1, start), min(end, len(lines)) + 1))

@tool
def grep(pattern: str, path: str = ".") -> str:
    """Search .py files for a regular expression. Returns file:line: text for up to 30 matches.

    Args:
        pattern: A Python regular expression, e.g. "def cart_total".
        path: Directory or file (relative to the repo root) to search.
    """
    base = _sb().resolve(path)
    rx = re.compile(pattern)
    files = [base] if base.is_file() else sorted(base.rglob("*.py"))
    hits = []
    for f in files:
        if "__pycache__" in f.parts: continue
        for n, line in enumerate(f.read_text().splitlines(), 1):
            if rx.search(line):
                hits.append(f"{f.relative_to(_sb().root)}:{n}: {line.strip()}")
    return "\n".join(hits[:30]) or "no matches"

@tool
def edit_file(path: str, old: str, new: str) -> str:
    """Replace one exact occurrence of `old` with `new` in a file. Fails if `old` is missing or appears more than once, so include enough surrounding text to be unique.

    Args:
        path: File path relative to the repo root. Files under tests/ cannot be edited.
        old: The exact existing text (including indentation) to replace.
        new: The replacement text.
    """
    p = _sb().writable(path)
    src = p.read_text()
    n = src.count(old)
    if old == new: raise ValueError("old and new are identical; nothing to change")
    if n == 0: raise ValueError("`old` text not found - re-read the file; whitespace must match exactly")
    if n > 1: raise ValueError(f"`old` matches {n} places - add surrounding lines to make it unique")
    p.write_text(src.replace(old, new))
    return f"edited {path}: replaced 1 occurrence ({len(old.splitlines())} line(s) -> {len(new.splitlines())} line(s))"

@tool
def run_tests(target: str = "") -> str:
    """Run the project's tests with pytest and return the (truncated) output. Run this after every edit.

    Args:
        target: Optional test file or node id, e.g. "tests/test_cart.py". Empty runs everything.
    """
    if target:
        _sb().resolve(target.split("::")[0])
    rc, out, _ = pytest_run(_sb().root, target)
    return (f"exit code {rc}\n" + out[-2500:]) if rc else "exit code 0\n" + out.strip().splitlines()[-1]

CODE_TOOLS = ToolBox(read_file, grep, edit_file, run_tests)
print([t["name"] for t in CODE_TOOLS.schemas()])

## 5 · Try the tools by hand (assert before you trust)
Before an LLM touches them, drive the tools directly on the first task's repo — exactly what a unit test for your agent's harness would do.

In [ ]:
t1 = TASKS[0]
SB["sb"] = Sandbox(WORKDIR / t1.id)

out, err = CODE_TOOLS.run("grep", {"pattern": "def apply_discount"});            print(out); assert not err and "pricing.py:1" in out
out, err = CODE_TOOLS.run("read_file", {"path": "shopkit/pricing.py", "start": 1, "end": 5}); print(out); assert "pct / 10" in out
out, err = CODE_TOOLS.run("edit_file", {"path": "shopkit/pricing.py", "old": "pct", "new": "percent"}); print("ambiguous edit ->", out); assert err and "matches" in out
out, err = CODE_TOOLS.run("edit_file", {"path": "shopkit/pricing.py", "old": "not in the file", "new": "x"}); print("missing edit   ->", out); assert err and "not found" in out
out, err = CODE_TOOLS.run("run_tests", {"target": "tests/test_pricing.py"}); print(out.splitlines()[0]); assert out.startswith("exit code 1") and "test_discount_quarter" in out

## 6 · Break it (part 1): escape attempts and tampering
A model can be wrong, confused, or manipulated by text in a file it reads (prompt injection — AG13 returns to this). So the **tool** must hold the line even if the model tries.
Each attempt below must be refused with a readable error, never a Python crash and never a leak.

In [ ]:
outside = pathlib.Path(tempfile.mkdtemp(prefix="ag12_outside_"))        # a "secret" the agent must never reach
(outside / "secret.txt").write_text("TOP-SECRET")
sb = _sb()
os.symlink(outside / "secret.txt", sb.root / "link_to_secret.txt")        # a symlink planted inside the repo
evil_sibling = sb.root.parent / (sb.root.name + "_evil")                  # a sibling dir whose name starts with the root's name
evil_sibling.mkdir(); (evil_sibling / "loot.txt").write_text("LOOT")

attempts = {
    "parent traversal":   "../../../../etc/passwd",
    "absolute path":      str(outside / "secret.txt"),
    "symlink inside repo": "link_to_secret.txt",
    "sibling-prefix dir": f"../{evil_sibling.name}/loot.txt",
}
for name, path in attempts.items():
    out, err = CODE_TOOLS.run("read_file", {"path": path})
    print(f"{name:<22} -> {'BLOCKED' if err else 'LEAKED!'}  {out[:70]}")
    assert err and "TOP-SECRET" not in out and "LOOT" not in out

naive_ok = str(evil_sibling / "loot.txt").startswith(str(sb.root))
print("\nnaive startswith() check would have allowed the sibling-prefix path:", naive_ok)
assert naive_ok is True      # why we use relative_to()

out, err = CODE_TOOLS.run("edit_file", {"path": "tests/test_pricing.py", "old": "== 150.0", "new": "== -300.0"})
print("edit a test file       ->", out); assert err and "read-only" in out
out, err = CODE_TOOLS.run("edit_file", {"path": "notes.txt", "old": "a", "new": "b"}); assert err

## 7 · Run the agent on one task (the test-driven loop)
The system prompt gives the **repo context** (a file tree) and the working method. The recorded trace is what a good run looks like: *reproduce → locate → read → minimal edit → re-run*.
The model never sees which file is buggy; it greps for the symbol the issue talks about. In OFFLINE mode the trace is scripted, **but the tools are real**: the edit changes a real file and the
last `run_tests` runs real pytest — if the script were wrong, the final assertion below would fail.

In [ ]:
def repo_tree(root: pathlib.Path) -> str:
    return "\n".join(str(p.relative_to(root)) for p in sorted(root.rglob("*.py")) if "__pycache__" not in p.parts)

def system_prompt(root):
    return ("You are a careful software engineer fixing a bug in a small Python library. Method: run the tests to reproduce, grep to locate the code, "
            "read only the lines you need, make the smallest possible edit with edit_file, then run the tests again. Never edit tests. "
            "When done, say what you changed in one or two sentences.\n\nRepository files:\n" + repo_tree(root))

def good_script(t: Task):
    return [call("run_tests", _say="Let me reproduce the failure first."),
            call("grep", _say="The issue points at this function; find it.", pattern=t.symbol),
            call("read_file", path=t.file, start=1, end=40),
            call("edit_file", _say="The expression is wrong; restoring the intended one.", path=t.file, old=t.buggy, new=t.correct),
            call("run_tests"),
            say(f"Fixed {t.id}: in {t.file} I changed `{t.buggy}` to `{t.correct}`. All tests pass.")]

def snapshot(root):
    return {str(p.relative_to(root)): p.read_text() for p in sorted(root.rglob("*.py")) if "__pycache__" not in p.parts}

def grade(task: Task, root: pathlib.Path) -> dict:
    """Independent verification: restore pristine tests, run pytest, check both groups."""
    for rel, text in TESTS.items():
        (root / rel).write_text(text)                            # tamper-proofing: the agent's test edits (if any) are discarded
    rc, out, res = pytest_run(root)
    f2p_ok = all(res.get(k) == "PASSED" for k in BASE[task.id]["f2p"])
    p2p_ok = all(res.get(k) == "PASSED" for k in BASE[task.id]["p2p"])
    return dict(resolved=f2p_ok and p2p_ok, f2p_ok=f2p_ok, p2p_ok=p2p_ok, rc=rc)

def changed_lines(before: dict, after: dict) -> int:
    n = 0
    for k in after:
        d = difflib.unified_diff(before[k].splitlines(), after[k].splitlines(), lineterm="", n=0)
        n += sum(1 for l in d if l[:1] in "+-" and l[:3] not in ("+++", "---"))
    return n

def solve(task: Task, script, max_turns: int = 10) -> dict:
    root = build_repo(task, WORKDIR / "runs")
    SB["sb"] = Sandbox(root)
    before = snapshot(root)
    res = run_agent(get_llm(script=script), CODE_TOOLS, f"Issue {task.id}: {task.issue}", system=system_prompt(root), max_turns=max_turns)
    verdict = grade(task, root)
    return dict(task=task.id, claims_fixed=bool(re.search(r"\bfix(ed)?\b", res.final_text.lower())), turns=res.turns, tool_calls=res.tracer.n_tool_calls(),
                edits=res.tracer.n_tool_calls("edit_file"), diff_lines=changed_lines(before, snapshot(root)), stop=res.stop, trace=res, **verdict)

r = solve(TASKS[0], good_script(TASKS[0]))
r["trace"].tracer.show()
print({k: v for k, v in r.items() if k != "trace"})
assert r["resolved"] and r["f2p_ok"] and r["p2p_ok"]
assert r["diff_lines"] == 2          # one line out, one line in: a minimal, reviewable patch
assert r["trace"].tracer.n_tool_calls("run_tests") == 2

### 7a · Read the patch like a reviewer
Because the edits are string replacements we can always reconstruct a unified diff. In a real workflow this diff — not the chat — is the artefact a human reviews and merges.

In [ ]:
root = WORKDIR / "runs" / TASKS[0].id
print("".join(difflib.unified_diff((WORKDIR / TASKS[0].id / TASKS[0].file).read_text().splitlines(True), (root / TASKS[0].file).read_text().splitlines(True),
                                   "a/" + TASKS[0].file, "b/" + TASKS[0].file)))

## 8 · Use it: the resolve-rate harness
Now measure like a benchmark does. For each task: fresh repo → run the agent → grade independently → record resolved / turns / tool calls / patch size. The headline metric is the **resolve rate**.
Per-task cost metrics matter too: a resolved task that took 40 tool calls is a different product from one that took 6.

In [ ]:
def evaluate(name, script_for):
    rows = [solve(t, script_for(t)) for t in TASKS]
    rate = sum(r["resolved"] for r in rows) / len(rows)
    print(f"\n== {name}: resolve rate {sum(r['resolved'] for r in rows)}/{len(rows)} = {rate:.0%}")
    print(f"{'task':<8}{'resolved':>9}{'F2P':>5}{'P2P':>5}{'claims fixed':>14}{'turns':>7}{'tools':>7}{'diff':>6}")
    for r in rows:
        print(f"{r['task']:<8}{str(r['resolved']):>9}{str(r['f2p_ok'])[0]:>5}{str(r['p2p_ok'])[0]:>5}{str(r['claims_fixed']):>14}{r['turns']:>7}{r['tool_calls']:>7}{r['diff_lines']:>6}")
    return rows, rate

rows, rate = evaluate("scripted good agent", good_script)
assert rate == 1.0 and all(r["diff_lines"] == 2 for r in rows)

## 9 · Break it (part 2): "done!" is not evidence
Two classic failure modes, scripted so you can study them:
* **Regression** — a fix that makes the failing test pass but deletes the input validation, silently breaking a PASS_TO_PASS test.
* **Test tampering** — the agent tries to make tests pass by editing them. The tool refuses; the agent then gives up *and says it succeeded anyway*.

Watch the columns "resolved" versus "claims fixed".

In [ ]:
def sloppy_script(t: Task):
    if t.id == "SK-101":      # regression: removes the validation guard together with the bug
        guard = ('    if not 0 <= pct <= 100:\n        raise ValueError("pct must be between 0 and 100")\n'
                 '    return round(price * (1 - pct / 10), 2)')
        return [call("run_tests"), call("edit_file", path=t.file, old=guard, new='    return round(price * (1 - pct / 100), 2)'),
                call("run_tests", target="tests/test_pricing.py::test_discount_quarter"), say("Fixed the discount bug. Done!")]
    if t.id == "SK-103":      # tamper with the test, get refused, claim success anyway
        return [call("run_tests"), call("edit_file", path="tests/test_paging.py", old="== [5]", new="== []"),
                say("Fixed - pagination now behaves as expected.")]
    return good_script(t)

bad_rows, bad_rate = evaluate("sloppy agent", sloppy_script)
assert [r["resolved"] for r in bad_rows] == [False, True, False]
assert all(r["claims_fixed"] for r in bad_rows), "the sloppy agent claims success on every task"
assert bad_rows[0]["f2p_ok"] and not bad_rows[0]["p2p_ok"]      # fixed the target test, broke another
tamper = next(e for e in bad_rows[2]["trace"].tracer.events if e.kind == "tool" and e.name == "edit_file")
assert tamper.detail["error"] and "read-only" in tamper.detail["result"]
print("\nself-reported success: 3/3   verified resolve rate:", f"{bad_rate:.0%}")

**Lessons.** (1) Report the *verified* resolve rate, never the agent's claim. (2) PASS_TO_PASS is not decoration — it is what catches overreaching "fixes". (3) The tool-level guard
(read-only tests) and the grader-level guard (pristine tests restored) are *both* worth having: defence in depth. (4) With real models, a few percentage points of resolve rate can be
noise on a small task set; use many tasks and repeated runs before you believe a comparison (AG14 builds the eval harness properly).

## 10 · From mini-agent to production tools: Claude Agent SDK, skills, hooks, subagents (overview)
Everything above was ~100 lines because we kept it small. Production coding agents add the same ingredients at scale. The **Claude Agent SDK** (Python and TypeScript; formerly the
*Claude Code SDK*) packages the agent harness that powers Claude Code so you can embed it instead of rebuilding it. At a conceptual level it gives you:

| Concept | What it is | How it relates to what you built |
|---|---|---|
| **Built-in tools** | File read/edit, shell, search (grep/glob) and more, with a permission system around them | our `read_file` / `edit_file` / `grep` / `run_tests`, plus approvals (AG09) |
| **Permissions / sandboxing** | Rules for which tools and paths the agent may use, ask-before-run modes | our `Sandbox` and protected `tests/` |
| **Hooks** | Your own code run at lifecycle points (e.g. before or after a tool call, when a session starts or ends), able to allow, block or annotate actions | *deterministic* enforcement: "block edits under `tests/`" or "run the tests after every edit" belongs in a hook, not in a prompt |
| **Skills** | A folder with a `SKILL.md` (name and description in front-matter, then instructions) plus optional scripts and resources; the model sees the short description and loads the rest only when relevant | reusable know-how ("how we write migrations") without bloating every prompt (AG05) |
| **Subagents** | Specialised assistants, each with its own context window, instructions and tool restrictions, to which the main agent delegates | AG11's isolated-context workers, e.g. a read-only "code explorer" |
| **Project memory** (`CLAUDE.md`) | A file of project conventions loaded as context | our repo-tree-in-the-system-prompt idea, grown up |
| **MCP servers** | External tools and data sources | AG10 |

This table is a *conceptual map*, not documentation: configuration formats, option names and event names change, so consult the current Claude Agent SDK and Claude Code docs before writing any.
The design questions, however, are the ones you just answered: which tools, which permissions, what is verified by code, and how is success measured?

## 11 · Exercises
**Practice (30 min)**
1. Add a fourth task, `SK-104`: seed a bug in `page_count` (e.g. replace `-(-n_items // size)` with `n_items // size`). You will need a new test in `TESTS` that exposes it; recompute `BASE` and write a scripted fix. Assert it resolves.
2. Make `grep` case-insensitive via an optional `ignore_case: bool = False` argument. Add an assertion that matches `DEF CART_TOTAL` only when the flag is set.

**Stretch (60 min)**
3. Add a **budget guard**: the harness stops a run after 8 tool calls and marks it `resolved=False, stop="budget"`. Write a looping script that triggers it.
4. Report `cost per resolved task` using `agentkit.trace.PRICES` and token usage from the tracer (give the scripted replies a non-zero `usage`, or run LIVE).

**Challenge (2 h, LIVE recommended)**
5. Run the same harness LIVE (just set `ANTHROPIC_API_KEY`; the system prompt and tools are unchanged). Pass `get_llm(script=...)` an empty-ish script — it is ignored in LIVE mode. Compare resolve rate and tool calls across two prompts (with and without the "never edit tests" rule). Did the agent *try* to edit tests when it was not told not to?

In [ ]:
# Starter for exercise 3 - a hard budget on tool calls. Fill in and enable the asserts.
def solve_with_budget(task: Task, script, max_tool_calls: int = 8) -> dict:
    # TODO: run like solve(), but if the tracer shows more than max_tool_calls tool calls, return resolved=False, stop="budget".
    # Hint: run_agent(max_turns=...) is a blunt version of this; counting tool calls is the precise one.
    raise NotImplementedError

# loop_script = [call("run_tests")] * 20
# r = solve_with_budget(TASKS[0], loop_script)
# assert r["stop"] in ("budget", "max_turns") and not r["resolved"]

## ✅ Checkpoint
1. Why does `edit_file` require `old` to match exactly once instead of letting the model rewrite the whole file?
2. A sandbox checks `str(path).startswith(str(root))`. Give an input that defeats it, and the fix.
3. The agent says "All tests pass" but your grader reports the task unresolved. Name two ways this can legitimately happen.
4. Why are PASS_TO_PASS tests part of the definition of "resolved"?

<details><summary>Answers</summary>

1. A small exact-match edit is cheap, reviewable as a diff, cannot silently drop code the model forgot to re-emit, and fails loudly when the model's picture of the file is stale (zero matches) or ambiguous (many matches), which the model can then repair.
2. A sibling directory whose name begins with the root's name (`/work/repo_evil` for root `/work/repo`) or a symlink inside the repo pointing outside. Fix: resolve symlinks first and use `Path.relative_to(root)` (or `os.path.commonpath`) on resolved paths.
3. It broke a PASS_TO_PASS test (regression) while running only the target test, or it edited the tests so they pass locally while the pristine tests fail; the agent's last test run may also have used stale or different tests than the grader.
4. Without them, deleting the validation or special-casing the failing input would count as a fix. "Resolved" means the bug is gone *and* nothing else broke.
</details>

## 📌 Key takeaways
- A coding agent is the AG02 loop plus **four sharp tools** and a **verifier**; the tests close the loop.
- **Edit by exact string replacement**, read by line range, give repo context up front, keep tests read-only.
- Sandbox with `resolve()` + `relative_to()`; file-tool checks are not an OS sandbox, so run untrusted code in a container.
- Measure **verified resolve rate** (FAIL_TO_PASS and PASS_TO_PASS), plus turns, tool calls, patch size and cost. An agent's own report is a claim, not a result.
- Production tooling (Claude Agent SDK, skills, hooks, subagents) packages these same ideas: put *enforcement* in hooks and tools, *know-how* in skills, *isolation* in subagents.

## 📚 Further reading
- Jimenez et al., *SWE-bench: Can Language Models Resolve Real-World GitHub Issues?* (2023), arXiv:2310.06770
- Yang et al., *SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering* (2024), arXiv:2405.15793
- Anthropic Engineering, *Building effective agents* (2024) — anthropic.com/engineering/building-effective-agents
- Anthropic docs: Claude Agent SDK, Claude Code (hooks, subagents, skills) — check the current documentation for configuration details

**Next → AG13 · Computer-Use & Browser Agents** (an observe→act loop on a simulated GUI, and the safety issues that come with reading untrusted screens)

In [ ]:
# Housekeeping: remove the scratch repos and the "outside" secret (everything lives in temp dirs we created)
shutil.rmtree(outside, ignore_errors=True)
WORK.cleanup()
print("cleaned up:", not WORKDIR.exists())